# NoLA / Colab: first matched anchor pilot
Upload this notebook in Colab. In the first code cell upload `NoLA-Colab-Inputs.zip` when prompted. Sections 1–4 run on CPU and do not require a GPU; don't leave a GPU connected just to run the audit. Section 5 is a separate real CLIP GPU throughput check; switch to a GPU only when you want to run it. Switching runtimes can clear uploaded files. No test split or paid job is launched automatically.

## 1. Settings and input
The comparison holds the saved CDE scores, original anchor quotas, adaptation rows, and validation rows fixed. Each class can exchange at most one anchor for an unselected candidate with the same predicted class. The candidate can't score more than 0.05 lower for that class. The labels in the CSV are used only to audit true correctness. These choices define a pilot, not a tuned recipe.

In [ ]:
from pathlib import Path
import hashlib, json, zipfile
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display

root = Path("/content") if Path("/content").exists() else Path.cwd()
archive = root / "NoLA-Colab-Inputs.zip"
inputs = root / "nola_inputs"
seeds = [42, 123, 2025]
head_epochs, head_batch, head_lr = 20, 8, 0.001
neighbors, min_support, max_score_drop = 5, 4, 0.05
bench_batch, warmup, repeats = 8, 3, 12

if not archive.exists():
    from google.colab import files
    uploaded = files.upload()
    archive.write_bytes(uploaded[archive.name])
with zipfile.ZipFile(archive) as bundle:
    bundle.extractall(inputs)
print("Inputs:", inputs)

## 2. Read the saved data and construct a visual graph
Match score rows to DINO feature rows by image path. Find cosine neighbors in the original 768-dimensional space, not PCA. Keep mutual links between images whose first-choice CLIP class agrees. Connected components are candidate visual groups, not verified semantic variations. Only groups with four or more images are eligible to gain a slot.

In [ ]:
from scipy.sparse.csgraph import connected_components

names = json.loads((inputs/"metrics.json").read_text())["class_order"]
by_class = {name: i for i, name in enumerate(names)}
features = np.load(inputs/"dino-pca.npz")
pred = pd.read_csv(inputs/"predictions.csv")
coords = pd.read_csv(inputs/"dino-pca-coordinates.csv")
original = pd.read_csv(inputs/"anchors.csv")

def aligned(part):
    frame = pred[(pred.stage == "cde") & (pred.split == part)].sort_values("row").reset_index(drop=True)
    keys = coords[coords.split.eq(part)].sort_values("row").image_path.to_numpy()
    assert np.array_equal(frame.image_path.to_numpy(), keys)
    return frame

train, val = aligned("adaptation"), aligned("validation")
x = torch.from_numpy(features["train_features"])
xval = torch.from_numpy(features["val_features"])
assert len(x)==len(train) and len(xval)==len(val)
prob = train[[f"prob_{i}" for i in range(len(names))]].to_numpy()
top1 = train.predicted_class.map(by_class).to_numpy()
y = train.true_class.map(by_class).to_numpy()
yval = torch.tensor(val.true_class.map(by_class).to_numpy())
base_ids = original.row.to_numpy(); base_labels = original.pseudo_class.map(by_class).to_numpy()
assert np.array_equal(train.iloc[base_ids].image_path.to_numpy(),original.path.to_numpy())

v = nn.functional.normalize(x,dim=1).numpy()
similarity = v @ v.T
np.fill_diagonal(similarity, -np.inf)
nearest = np.argsort(-similarity, axis=1, kind="stable")[:, :neighbors]
a = np.zeros_like(similarity,dtype=bool)
a[np.arange(len(x))[:,None], nearest] = True
graph = a & a.T & (top1[:,None] == top1)
_, group = connected_components(graph,directed=False)
supported = {g for g,n in enumerate(np.bincount(group)) if n>=min_support}
print(f"{len(train)} adaptation / {len(val)} validation rows; {len(supported)} supported groups")

## 3. Exchange an anchor only when coverage increases
For every class, look for an unrepresented supported group. A donor must be a first-choice anchor from a group with at least two anchors of that class; keep the existing fallback cases as-is. Only accept candidates that add a group and meet the confidence gate. No true labels enter the decision.

In [ ]:
new_ids = base_ids.copy()
used = set(base_ids.tolist())
changes = []
for cls in range(len(names)):
    slots = np.flatnonzero(base_labels == cls)
    counts = pd.Series(group[new_ids[slots]]).value_counts().to_dict()
    donors = [k for k in slots if top1[new_ids[k]]==cls and counts[group[new_ids[k]]]>=2]
    candidates = [i for i in np.flatnonzero(top1==cls) if i not in used and group[i] in supported and group[i] not in counts]
    valid = [(k,i) for k in donors for i in candidates if prob[i,cls]>=prob[new_ids[k],cls]-max_score_drop]
    if valid:
        k,i = min(valid,key=lambda pair:(prob[new_ids[pair[0]],cls]-prob[pair[1],cls], pair[1],pair[0]))
        donor = int(new_ids[k]); new_ids[k]=i; used.add(i)
        changes.append(dict(class_name=names[cls], removed=donor, added=i, old_score=prob[donor,cls], new_score=prob[i,cls]))
assert len(set(new_ids.tolist()))==len(base_ids)

def coverage(ids):
    return len({group[i] for i,c in zip(ids,base_labels) if top1[i]==c and group[i] in supported})

assert coverage(new_ids)>=coverage(base_ids)
print("Swaps:",len(changes))
if changes:
    display(pd.DataFrame(changes).assign(removed_correct=lambda d:[y[i]==top1[i] for i in d.removed], added_correct=lambda d:[y[i]==top1[i] for i in d.added]))
print("Original: covered groups / correct anchors:",coverage(base_ids),int((y[base_ids]==base_labels).sum()))
print("Coverage selector: covered groups / correct anchors:",coverage(new_ids),int((y[new_ids]==base_labels).sum()))

## 4. Compare matched DINO heads
Use the same head architecture, seed, batch size and epoch budget. Read differences as a small validation diagnostic. If no swaps pass the gate, the two selectors should give identical scores; that is a valid outcome. This is not a prompted CLIP result.

In [ ]:
def fit(ids, seed):
    torch.manual_seed(seed)
    order = torch.Generator().manual_seed(seed)
    head = nn.Sequential(nn.Linear(x.shape[1],256),nn.ReLU(),nn.Linear(256,len(names)))
    opt = torch.optim.Adam(head.parameters(),lr=head_lr)
    labels = torch.tensor(base_labels)
    for _ in range(head_epochs):
        for batch in torch.randperm(len(ids),generator=order).split(head_batch):
            opt.zero_grad()
            nn.functional.cross_entropy(head(x[ids[batch.numpy()]]),labels[batch]).backward()
            opt.step()
    with torch.no_grad():
        return int((head(xval).argmax(1)==yval).sum())

results = pd.DataFrame([dict(seed=s, confidence_topk=fit(base_ids,s), visual_coverage=fit(new_ids,s)) for s in seeds])
results["delta_correct"] = results.visual_coverage - results.confidence_topk
print(f"Correct out of {len(val)} validation images")
display(results)
results.to_csv(root/"nola_anchor_pilot_validation.csv",index=False)

## 5. Optional: time real CLIP images on the current GPU
Switch to a GPU runtime, rerun §1, then run only this section. It downloads the public ViT-B/32 checkpoint once (~354 MB), checks its SHA-256, and times batched inference on real EuroSAT anchor JPEGs. This measures **inference throughput**, not prompted training time or Colab units consumed. Read your units balance separately and disconnect when finished.

In [ ]:
import importlib.util
import time
import urllib.request
from PIL import Image
from torchvision import transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    print("No GPU connected; switch runtime type to GPU to measure throughput.")
else:
    torch.cuda.reset_peak_memory_stats()
    url = (inputs/"clip-weight-url.txt").read_text().strip()
    ckpt = root/"ViT-B-32.pt"
    if not ckpt.exists():
        print("Downloading public CLIP weights…",flush=True)
        partial = ckpt.with_suffix(".part")
        with urllib.request.urlopen(url,timeout=30) as src, partial.open("wb") as dst:
            import shutil
            shutil.copyfileobj(src,dst)
        with partial.open("rb") as f:
            assert hashlib.file_digest(f,"sha256").hexdigest()==url.split("/")[-2], "Downloaded weights failed verification."
        partial.replace(ckpt)
    with ckpt.open("rb") as f:
        assert hashlib.file_digest(f,"sha256").hexdigest()==url.split("/")[-2], "Invalid weights; delete the cached file and retry."
    spec = importlib.util.spec_from_file_location("pinned_clip_model",inputs/"clip_model.py")
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    checkpoint = torch.jit.load(str(ckpt),map_location="cpu")
    model = module.build_model(checkpoint.state_dict()).to(device).eval()
    del checkpoint
    preprocess = T.Compose([T.Resize(224,interpolation=T.InterpolationMode.BICUBIC),T.CenterCrop(224),T.ToTensor(),T.Normalize((.48145466,.4578275,.40821073),(.26862954,.26130258,.27577711))])
    batch = []
    for path in sorted((inputs/"images").glob("*.jpg"))[:bench_batch]:
        with Image.open(path) as im: batch.append(preprocess(im.convert("RGB")))
    batch = torch.stack(batch).to(device)
    with torch.inference_mode():
        for _ in range(warmup): model.encode_image(batch)
        torch.cuda.synchronize(); start = time.perf_counter()
        for _ in range(repeats): vectors = model.encode_image(batch)
        torch.cuda.synchronize(); elapsed = time.perf_counter()-start
    assert vectors.shape==(len(batch),512)
    print(torch.cuda.get_device_name(), f"| {len(batch)*repeats/elapsed:.1f} images/s | peak {torch.cuda.max_memory_allocated()/2**20:.0f} MiB allocated")